# Comprehensive Demo: Feature Attribution Methods

This notebook demonstrates all implemented feature attribution methods:
- Integrated Gradients
- LIME (Local Interpretable Model-agnostic Explanations)
- SHAP (SHapley Additive exPlanations)
- GradCAM (Gradient-weighted Class Activation Mapping)

We'll test each method on:
- ResNet50 (Computer Vision)
- Vision Transformer (Computer Vision)
- GPT-2 (Natural Language Processing)

In [1]:
# Setup paths and imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Import all explainers
from src.exlib.new_explainers import (
    IntGradImage, IntGradText, IntGradExplanation,
    LimeImage, LimeText, LimeExplanation,
    ShapImage, ShapText, ShapExplanation,
    GradCAMImage, GradCAMText, GradCAMExplanation
)

from src.exlib.new_explainers.utils.masking import patch_segment_image

# Import test utilities
from tests.fixtures import (
    get_vision_model, get_text_model,
    get_test_image, get_test_text_inputs
)

print("All imports successful!")

All imports successful!


## Visualization Functions

In [2]:
x = torch.rand(1,3,9,9)

In [6]:
patch_segment_image(x, 4)

tensor([[[0, 0, 0, 1, 1, 1, 2, 2, 2],
         [0, 0, 0, 1, 1, 1, 2, 2, 2],
         [0, 0, 0, 1, 1, 1, 2, 2, 2],
         [3, 3, 3, 4, 4, 4, 5, 5, 5],
         [3, 3, 3, 4, 4, 4, 5, 5, 5],
         [3, 3, 3, 4, 4, 4, 5, 5, 5],
         [6, 6, 6, 7, 7, 7, 8, 8, 8],
         [6, 6, 6, 7, 7, 7, 8, 8, 8],
         [6, 6, 6, 7, 7, 7, 8, 8, 8]]])

In [ ]:
def visualize_image_attribution(image, attribution, title="Attribution", cmap='RdBu_r'):
    """Visualize image attribution heatmap."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    img_display = image.permute(1, 2, 0).detach().cpu().numpy()
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
    ax1.imshow(img_display)
    ax1.set_title("Original Image")
    ax1.axis('off')
    
    # Attribution heatmap (average across channels)
    attr_avg = attribution.mean(dim=0).detach().cpu().numpy()
    im = ax2.imshow(attr_avg, cmap=cmap)
    ax2.set_title(f"{title} Heatmap")
    ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046)
    
    # Overlay
    ax3.imshow(img_display)
    ax3.imshow(attr_avg, cmap=cmap, alpha=0.5)
    ax3.set_title("Overlay")
    ax3.axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_text_attribution(tokens, attribution, title="Attribution"):
    """Visualize text attribution as bar chart."""
    # Convert to numpy
    attr_values = attribution.detach().cpu().numpy()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Create bar chart
    x = np.arange(len(tokens))
    colors = ['red' if v < 0 else 'green' for v in attr_values]
    bars = ax.bar(x, attr_values, color=colors, alpha=0.7)
    
    # Customize
    ax.set_xticks(x)
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_xlabel('Tokens')
    ax.set_ylabel('Attribution Score')
    ax.set_title(f'{title} - Token Importance')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

print("Visualization functions defined!")

## Load Models

In [ ]:
# Load vision models
print("Loading ResNet50...")
resnet_model, resnet_config = get_vision_model("resnet50")
print(f"ResNet50 loaded! Input size: {resnet_config['input_size']}")

print("\nLoading Vision Transformer...")
vit_model, vit_config = get_vision_model("vit")
print(f"ViT loaded! Input size: {vit_config['input_size']}")

# Load text model
print("\nLoading GPT-2...")
gpt2_model, gpt2_config = get_text_model("gpt2")
tokenizer = gpt2_config["tokenizer"]
print("GPT-2 loaded!")

## Prepare Test Inputs

In [ ]:
# Get test images
resnet_image = get_test_image(size=resnet_config["input_size"])
vit_image = get_test_image(size=vit_config["input_size"])

# Get test text
test_text = "The movie was absolutely fantastic, with great acting and stunning visuals."
inputs = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
input_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"ResNet image shape: {resnet_image.shape}")
print(f"ViT image shape: {vit_image.shape}")
print(f"Text tokens ({len(tokens)}): {tokens}")

## 1. Integrated Gradients

In [ ]:
# ResNet50 with IntGrad
print("Running Integrated Gradients on ResNet50...")
intgrad_image = IntGradImage(n_steps=50)
intgrad_resnet_exp = intgrad_image.explain(resnet_model, resnet_image)
print(f"Target class: {intgrad_resnet_exp.metadata['target']}")
visualize_image_attribution(resnet_image, intgrad_resnet_exp.attributions, "IntGrad (ResNet50)")

In [ ]:
# ViT with IntGrad
print("Running Integrated Gradients on ViT...")
intgrad_vit_exp = intgrad_image.explain(vit_model, vit_image)
print(f"Target class: {intgrad_vit_exp.metadata['target']}")
visualize_image_attribution(vit_image, intgrad_vit_exp.attributions, "IntGrad (ViT)")

In [ ]:
# GPT-2 with IntGrad
print("Running Integrated Gradients on GPT-2...")
intgrad_text = IntGradText(embedding_layer=gpt2_config["embedding_layer"], n_steps=50)
intgrad_text_exp = intgrad_text.explain(gpt2_model, input_ids)
print(f"Target class: {intgrad_text_exp.metadata['target']}")
visualize_text_attribution(tokens, intgrad_text_exp.attributions, "IntGrad (GPT-2)")

## 2. LIME

In [ ]:
# ResNet50 with LIME
print("Running LIME on ResNet50...")
lime_image = LimeImage(n_samples=100, n_segments=16)
lime_resnet_exp = lime_image.explain(resnet_model, resnet_image, return_segments=True)
print(f"Target class: {lime_resnet_exp.metadata['target']}")
print(f"R² score: {lime_resnet_exp.r2_score:.4f}")
visualize_image_attribution(resnet_image, lime_resnet_exp.attributions, "LIME (ResNet50)")

In [ ]:
lime_resnet_exp.attributions.min(), lime_resnet_exp.attributions.max()

In [ ]:
# Visualize LIME segments
if lime_resnet_exp.segments is not None:
    plt.figure(figsize=(8, 8))
    plt.imshow(lime_resnet_exp.segments.cpu().numpy(), cmap='tab20')
    plt.title("LIME Segmentation")
    plt.colorbar()
    plt.axis('off')
    plt.show()

In [ ]:
# GPT-2 with LIME
print("Running LIME on GPT-2...")
lime_text = LimeText(
    embedding_layer=gpt2_config["embedding_layer"],
    n_samples=100,
    mask_token_id=tokenizer.pad_token_id or 0
)
lime_text_exp = lime_text.explain(gpt2_model, input_ids)
print(f"Target class: {lime_text_exp.metadata['target']}")
print(f"R² score: {lime_text_exp.r2_score:.4f}")
visualize_text_attribution(tokens, lime_text_exp.attributions, "LIME (GPT-2)")

## 3. SHAP

In [ ]:
# ResNet50 with SHAP
print("Running SHAP on ResNet50...")
shap_image = ShapImage(n_samples=100, n_segments=10)
shap_resnet_exp = shap_image.explain(resnet_model, resnet_image)
print(f"Target class: {shap_resnet_exp.metadata['target']}")
print(f"Expected value: {shap_resnet_exp.expected_value:.4f}")
print(f"Base value: {shap_resnet_exp.base_value:.4f}")
visualize_image_attribution(resnet_image, shap_resnet_exp.attributions, "SHAP (ResNet50)")

In [ ]:
# GPT-2 with SHAP
print("Running SHAP on GPT-2...")
shap_text = ShapText(
    embedding_layer=gpt2_config["embedding_layer"],
    n_samples=100,
    mask_token_id=tokenizer.pad_token_id or 0
)
shap_text_exp = shap_text.explain(gpt2_model, input_ids)
print(f"Target class: {shap_text_exp.metadata['target']}")
print(f"Expected value: {shap_text_exp.expected_value:.4f}")
print(f"Base value: {shap_text_exp.base_value:.4f}")
visualize_text_attribution(tokens, shap_text_exp.attributions, "SHAP (GPT-2)")

## 4. GradCAM

In [ ]:
# ResNet50 with GradCAM
print("Running GradCAM on ResNet50...")
gradcam_image = GradCAMImage()  # Auto-select layer
gradcam_resnet_exp = gradcam_image.explain(resnet_model, resnet_image)
print(f"Target class: {gradcam_resnet_exp.metadata['target']}")
print(f"Selected layer: {gradcam_resnet_exp.metadata['layer_name']}")
visualize_image_attribution(resnet_image, gradcam_resnet_exp.attributions, "GradCAM (ResNet50)")

In [ ]:
# GPT-2 with GradCAM (embedding gradients)
print("Running GradCAM on GPT-2...")
gradcam_text = GradCAMText(embedding_layer=gpt2_config["embedding_layer"])
gradcam_text_exp = gradcam_text.explain(gpt2_model, input_ids)
print(f"Target class: {gradcam_text_exp.metadata['target']}")
print(f"Method: {gradcam_text_exp.metadata['method']}")
visualize_text_attribution(tokens, gradcam_text_exp.attributions, "GradCAM (GPT-2)")

## Comparison: All Methods Side by Side

In [ ]:
# Compare all vision methods on ResNet50
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Original image
img_display = resnet_image.permute(1, 2, 0).detach().cpu().numpy()
img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())

# Plot original
axes[0, 0].imshow(img_display)
axes[0, 0].set_title("Original Image", fontsize=14)
axes[0, 0].axis('off')
axes[1, 0].axis('off')

# Plot attributions
methods = [
    ("IntGrad", intgrad_resnet_exp.attributions),
    ("LIME", lime_resnet_exp.attributions),
    ("SHAP", shap_resnet_exp.attributions),
    ("GradCAM", gradcam_resnet_exp.attributions)
]

for i, (name, attr) in enumerate(methods, 1):
    if i < 4:
        ax = axes[0, i]
    else:
        ax = axes[1, i-3]
    
    attr_avg = attr.mean(dim=0).detach().cpu().numpy()
    im = ax.imshow(attr_avg, cmap='RdBu_r')
    ax.set_title(name, fontsize=14)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Hide unused
axes[1, 1].axis('off')
axes[1, 2].axis('off')
axes[1, 3].axis('off')

plt.suptitle("Feature Attribution Methods Comparison (ResNet50)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Compare text methods
fig, axes = plt.subplots(4, 1, figsize=(15, 16))

text_methods = [
    ("IntGrad", intgrad_text_exp.attributions),
    ("LIME", lime_text_exp.attributions),
    ("SHAP", shap_text_exp.attributions),
    ("GradCAM", gradcam_text_exp.attributions)
]

for i, (name, attr) in enumerate(text_methods):
    ax = axes[i]
    attr_values = attr.detach().cpu().numpy()
    
    x = np.arange(len(tokens))
    colors = ['red' if v < 0 else 'green' for v in attr_values]
    bars = ax.bar(x, attr_values, color=colors, alpha=0.7)
    
    ax.set_xticks(x)
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_ylabel('Attribution Score')
    ax.set_title(f'{name} - Token Importance', fontsize=14)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.suptitle(f'Text Attribution Methods Comparison (GPT-2)\n"{test_text}"', fontsize=16)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Compute statistics for each method
print("Attribution Statistics (ResNet50)")
print("=" * 50)

for name, attr in methods:
    attr_flat = attr.flatten().detach().cpu().numpy()
    print(f"\n{name}:")
    print(f"  Mean: {attr_flat.mean():.6f}")
    print(f"  Std:  {attr_flat.std():.6f}")
    print(f"  Min:  {attr_flat.min():.6f}")
    print(f"  Max:  {attr_flat.max():.6f}")
    print(f"  % Positive: {(attr_flat > 0).mean() * 100:.1f}%")

print("\n" + "=" * 50)
print("Attribution Statistics (GPT-2)")
print("=" * 50)

for name, attr in text_methods:
    attr_values = attr.detach().cpu().numpy()
    print(f"\n{name}:")
    print(f"  Mean: {attr_values.mean():.6f}")
    print(f"  Std:  {attr_values.std():.6f}")
    print(f"  Min:  {attr_values.min():.6f}")
    print(f"  Max:  {attr_values.max():.6f}")
    print(f"  % Positive: {(attr_values > 0).mean() * 100:.1f}%")

## Method-Specific Features

In [ ]:
# Demonstrate LIME's local model quality
print("LIME R² Scores (Model Fit Quality):")
print(f"  ResNet50: {lime_resnet_exp.r2_score:.4f}")
print(f"  GPT-2: {lime_text_exp.r2_score:.4f}")

# Demonstrate SHAP's expected values
print("\nSHAP Expected Values:")
print(f"  ResNet50: {shap_resnet_exp.expected_value:.4f} (base: {shap_resnet_exp.base_value:.4f})")
print(f"  GPT-2: {shap_text_exp.expected_value:.4f} (base: {shap_text_exp.base_value:.4f})")

# Demonstrate IntGrad convergence
print("\nIntegrated Gradients Convergence Check:")
intgrad_conv = intgrad_image.explain(resnet_model, resnet_image, return_convergence_delta=True)
print(f"  Convergence delta: {intgrad_conv.convergence_delta:.6f}")
print(f"  (Lower is better, < 1.0 is good)")

## Performance Comparison

In [ ]:
import time

# Benchmark execution times
print("Execution Time Comparison (ResNet50)")
print("=" * 40)

# IntGrad
start = time.time()
_ = IntGradImage(n_steps=20).explain(resnet_model, resnet_image)
intgrad_time = time.time() - start
print(f"IntGrad (20 steps): {intgrad_time:.2f}s")

# LIME
start = time.time()
_ = LimeImage(n_samples=50, n_segments=10).explain(resnet_model, resnet_image)
lime_time = time.time() - start
print(f"LIME (50 samples): {lime_time:.2f}s")

# SHAP
start = time.time()
_ = ShapImage(n_samples=50, n_segments=10).explain(resnet_model, resnet_image)
shap_time = time.time() - start
print(f"SHAP (50 samples): {shap_time:.2f}s")

# GradCAM
start = time.time()
_ = GradCAMImage().explain(resnet_model, resnet_image)
gradcam_time = time.time() - start
print(f"GradCAM: {gradcam_time:.2f}s")

print(f"\nFastest: GradCAM ({gradcam_time:.2f}s)")
print(f"Slowest: {max([(intgrad_time, 'IntGrad'), (lime_time, 'LIME'), (shap_time, 'SHAP')], key=lambda x: x[0])[1]}")

## Conclusion

This notebook demonstrated all four implemented feature attribution methods:

1. **Integrated Gradients**: Gradient-based method that integrates gradients along a path from baseline to input
2. **LIME**: Model-agnostic method using local linear approximations
3. **SHAP**: Game-theoretic approach based on Shapley values
4. **GradCAM**: Gradient-based method specifically designed for CNNs

Each method has its strengths:
- **Speed**: GradCAM is fastest, followed by IntGrad
- **Model-agnostic**: LIME and SHAP work with any model
- **Theoretical guarantees**: SHAP provides Shapley value properties
- **Visual quality**: GradCAM often provides smooth, interpretable heatmaps for CNNs

All methods follow the same low-abstraction API pattern:
```python
explainer = MethodName(configs)
explanation = explainer.explain(model, input, target)
```